anndata version

In [1]:
!pip install anndata==0.8.0
#this needs to be done if error below is seem:
#AnnDataReadError: Above error raised while reading key '/obsm' of type <class 'h5py._hl.group.Group'> from /.

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 14.5 MB/s eta 0:00:0000:010:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


Import libraries

In [2]:
import anndata as ad
import scanpy as sc
import scipy.io as sio
import pandas as pd
import os
import glob

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import anndata files

In [3]:
#show work directory
os.getcwd()

'/scratch/data/docker/SAMapH5ad'

Function Anndata_to_Seurat()

In [4]:
def Anndata_to_Seurat(filename, org):
    """
    Processes an AnnData file, creates a directory named after the file,
    and saves the count matrix, cell barcodes, gene IDs, and metadata
    into that directory.

    Args:
        filename (str): The path to the input AnnData file (.h5ad).
        org (str): The organism name, used as a prefix for the output files.
    """
    print("Processing file:", filename)

    # Extract the base name of the file to use as the folder name
    folder_name = os.path.splitext(os.path.basename(filename))[0]

    # Create the directory; exist_ok=True prevents an error if the directory already exists
    try:
        os.makedirs(folder_name, exist_ok=True)
        print(f"Successfully created directory: {folder_name}")
    except OSError as e:
        print(f"Error creating directory {folder_name}: {e}")
        return

    # Read the AnnData file
    try:
        scdata = sc.read_h5ad(filename=filename)
    except Exception as e:
        print(f"Error reading AnnData file: {e}")
        return

    print("Organism name:", org)

    # Define file paths within the new directory
    mtx_path = os.path.join(folder_name, f"scdata_{org}.mtx")
    gene_id_path = os.path.join(folder_name, f"geneID_{org}.csv")
    cell_id_path = os.path.join(folder_name, f"cellID_{org}.csv")
    metadata_path = os.path.join(folder_name, f"metadata_{org}.csv")

    # Export count matrix as a sparse matrix
    try:
        sio.mmwrite(mtx_path, scdata.X)
        print(f"Count matrix saved to: {mtx_path}")
    except Exception as e:
        print(f"Error saving count matrix: {e}")

    # Export gene IDs
    try:
        gene_df = pd.DataFrame(scdata.var_names)
        gene_df.to_csv(gene_id_path, index=False, header=False)
        print(f"Gene IDs saved to: {gene_id_path}")
    except Exception as e:
        print(f"Error saving gene IDs: {e}")

    # Export cell barcodes
    try:
        cell_df = pd.DataFrame(scdata.obs_names)
        cell_df.to_csv(cell_id_path, index=False, header=False)
        print(f"Cell IDs saved to: {cell_id_path}")
    except Exception as e:
        print(f"Error saving cell IDs: {e}")

    # Export metadata (annotation)
    try:
        scdata.obs.to_csv(metadata_path)
        print(f"Metadata saved to: {metadata_path}")
    except Exception as e:
        print(f"Error saving metadata: {e}")

In [5]:
# convert individual files, not run unless needed
Anndata_to_Seurat("DR_dat_nounlabeled_06232026_yw.h5ad", "DR")
Anndata_to_Seurat("AC_dat_nounlabeled_06232026_yw.h5ad", "AC")
Anndata_to_Seurat("XT_dat_nounlabeled_06232026_yw.h5ad", "XT")
Anndata_to_Seurat("MO_dat_nounlabeled_06232026_yw.h5ad", "MO")

Processing file: DR_07132026.h5ad
Successfully created directory: DR_07132026
Organism name: DR
Count matrix saved to: DR_07132026/scdata_DR.mtx
Gene IDs saved to: DR_07132026/geneID_DR.csv
Cell IDs saved to: DR_07132026/cellID_DR.csv
Metadata saved to: DR_07132026/metadata_DR.csv
